# Proyecto 2 - Modelo predictivo Saber 11 Atlántico

**Estudiante:** Sofía Vásquez (202123910)  
**Rol:** Ciencia de datos  
**Usuario final:** Secretaría de Educación del Atlántico  

## Pregunta de negocio

**¿Es posible predecir si un estudiante del departamento del Atlántico obtendrá un puntaje global alto en las pruebas Saber 11 a partir de sus características socioeconómicas, familiares e institucionales?**

Este notebook desarrolla un modelo de clasificación binaria basado en redes neuronales para predecir si un estudiante alcanza un desempeño alto en Saber 11.


In [2]:
import numpy as np
import pandas as pd

## 2. Cargar datos

Usa el archivo limpio que generaste después de la limpieza y el One-Hot Encoding.  
Si tu archivo tiene otro nombre, cambia `DATA_PATH`.


In [4]:
DATA_PATH = "../Tarea 2 - Limpieza_Datos/saber11_limpio.csv"

df = pd.read_csv(DATA_PATH)

## 3. Definir variable objetivo

Se define como **alto desempeño** obtener un puntaje global mayor o igual a 300.

- `1`: alto desempeño  
- `0`: no alto desempeño


In [ ]:
score_col = "punt_global"

if score_col not in df.columns:
    raise ValueError("No se encontró la columna 'punt_global'. Revisa el nombre de la columna en tu archivo.")

df[score_col] = pd.to_numeric(df[score_col], errors="coerce")
df = df.dropna(subset=[score_col])

df["alto_desempeno"] = (df[score_col] >= 300).astype(int)

print("Distribución de la variable objetivo:")
print(df["alto_desempeno"].value_counts())
print("\nDistribución porcentual:")
print(df["alto_desempeno"].value_counts(normalize=True))


## 4. Selección de variables familiares

El modelo solo usará variables familiares:

- Estrato de vivienda
- Educación de la madre
- Educación del padre
- Personas en el hogar
- Cuartos del hogar
- Automóvil
- Lavadora

El código funciona en dos casos:

1. Si el archivo todavía tiene las columnas originales, hace limpieza y One-Hot Encoding.
2. Si el archivo ya tiene las columnas codificadas, las detecta automáticamente.


In [ ]:
variables_familiares = [
    'fami_estratovivienda',
    'fami_educacionmadre',
    'fami_educacionpadre',
    'fami_personashogar',
    'fami_cuartoshogar',
    'fami_tieneautomovil',
    'fami_tienelavadora'
]

variables_originales_existentes = [
    col for col in variables_familiares
    if col in df.columns
]

print("Variables familiares originales encontradas:")
print(variables_originales_existentes)


## 5. Limpieza y One-Hot Encoding de variables familiares

Esta celda solo aplica el One-Hot Encoding si las variables originales todavía existen.  
Si ya estaban codificadas en el archivo limpio, no repite el proceso.


In [ ]:
if len(variables_originales_existentes) > 0:

    # Limpieza de texto
    for col in variables_originales_existentes:
        df.loc[:, col] = (
            df[col]
            .astype(str)
            .str.lower()
            .str.strip()
        )

    # Reemplazar valores problemáticos
    valores_vacios = ['', ' ', 'nan', 'none', 'null']

    df.loc[:, variables_originales_existentes] = (
        df[variables_originales_existentes]
        .replace(valores_vacios, pd.NA)
    )

    # Imputar faltantes
    for col in variables_originales_existentes:
        df.loc[:, col] = df[col].fillna('sin_informacion')

    # One-Hot Encoding
    df = pd.get_dummies(
        df,
        columns=variables_originales_existentes,
        prefix=variables_originales_existentes,
        drop_first=False
    )

# Detectar columnas familiares codificadas
columnas_familiares_encoded = [
    col for col in df.columns
    if col.startswith(tuple([v + "_" for v in variables_familiares]))
]

if len(columnas_familiares_encoded) == 0:
    raise ValueError("No se encontraron columnas familiares codificadas. Revisa la limpieza o el archivo cargado.")

# Convertir True/False a 0/1
df[columnas_familiares_encoded] = df[columnas_familiares_encoded].astype(int)

print("Número de columnas familiares codificadas:", len(columnas_familiares_encoded))
columnas_familiares_encoded[:20]


## 6. Construir matriz de variables predictoras

Se usa únicamente el conjunto de columnas familiares codificadas.


In [ ]:
X = df[columnas_familiares_encoded].copy()
y = df["alto_desempeno"].copy()

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

X.head()


## 7. División entrenamiento / prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("\nDistribución en entrenamiento:")
print(y_train.value_counts(normalize=True))


## 8. Modelo base: Regresión logística

Este modelo sirve como línea base para comparar el desempeño de la red neuronal.


In [ ]:
baseline = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)

baseline.fit(X_train, y_train)

y_pred_base = baseline.predict(X_test)
y_proba_base = baseline.predict_proba(X_test)[:, 1]

baseline_metrics = {
    "accuracy": accuracy_score(y_test, y_pred_base),
    "precision": precision_score(y_test, y_pred_base, zero_division=0),
    "recall": recall_score(y_test, y_pred_base, zero_division=0),
    "f1": f1_score(y_test, y_pred_base, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_proba_base)
}

baseline_metrics


## 9. Red neuronal

Se propone una red neuronal sencilla para clasificación binaria.


In [ ]:
def construir_modelo(input_dim, capa_1=64, capa_2=32, dropout_1=0.30, dropout_2=0.20, learning_rate=0.001):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(capa_1, activation="relu"),
        layers.Dropout(dropout_1),
        layers.Dense(capa_2, activation="relu"),
        layers.Dropout(dropout_2),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.AUC(name="auc")]
    )

    return model

model = construir_modelo(input_dim=X_train.shape[1])
model.summary()


## 10. Entrenamiento del modelo

In [ ]:
params = {
    "capa_1": 64,
    "capa_2": 32,
    "dropout_1": 0.30,
    "dropout_2": 0.20,
    "learning_rate": 0.001,
    "epochs": 30,
    "batch_size": 128
}

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=params["epochs"],
    batch_size=params["batch_size"],
    callbacks=[early_stop],
    verbose=1
)


## 11. Evaluación del modelo

In [ ]:
y_proba_nn = model.predict(X_test).ravel()
y_pred_nn = (y_proba_nn >= 0.5).astype(int)

nn_metrics = {
    "accuracy": accuracy_score(y_test, y_pred_nn),
    "precision": precision_score(y_test, y_pred_nn, zero_division=0),
    "recall": recall_score(y_test, y_pred_nn, zero_division=0),
    "f1": f1_score(y_test, y_pred_nn, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_proba_nn)
}

nn_metrics


## 12. Comparación de modelos

In [ ]:
comparison = pd.DataFrame([
    {"modelo": "Regresión logística", **baseline_metrics},
    {"modelo": "Red neuronal", **nn_metrics}
])

comparison


## 13. Matriz de confusión

In [ ]:
cm = confusion_matrix(y_test, y_pred_nn)

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Matriz de confusión - Red neuronal")
plt.xlabel("Predicción")
plt.ylabel("Valor real")
plt.xticks([0, 1], ["No alto", "Alto"])
plt.yticks([0, 1], ["No alto", "Alto"])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.show()

print(classification_report(y_test, y_pred_nn, target_names=["No alto desempeño", "Alto desempeño"]))


## 14. Curva ROC

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba_nn)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"Red neuronal AUC = {nn_metrics['roc_auc']:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("Tasa de falsos positivos")
plt.ylabel("Tasa de verdaderos positivos")
plt.title("Curva ROC")
plt.legend()
plt.show()


## 15. Registro de experimento con MLflow

Si MLflow está instalado, se registran parámetros, métricas y modelo.


In [ ]:
if MLFLOW_AVAILABLE:
    mlflow.set_experiment("Saber11_Atlantico_Familiares_Clasificacion")

    with mlflow.start_run(run_name="modelo_familiar_sofia"):
        mlflow.log_params(params)
        mlflow.log_param("numero_variables_encoded", X_train.shape[1])

        for metric_name, metric_value in nn_metrics.items():
            mlflow.log_metric(metric_name, metric_value)

        mlflow.keras.log_model(model, "modelo_red_neuronal_familiar")

    print("Experimento registrado en MLflow.")
else:
    print("MLflow no está disponible en este ambiente.")


## 16. Guardar modelo y columnas para el tablero

Se guardan:
- modelo entrenado
- columnas usadas por el modelo

Estas columnas son necesarias para que Dash construya el mismo input.


In [ ]:
os.makedirs("models", exist_ok=True)

model.save("models/modelo_familiar_alto_desempeno.keras")
joblib.dump(columnas_familiares_encoded, "models/columnas_familiares_encoded.pkl")

print("Modelo guardado en: models/modelo_familiar_alto_desempeno.keras")
print("Columnas guardadas en: models/columnas_familiares_encoded.pkl")


## 17. Conclusión preliminar

El modelo permite estimar la probabilidad de que un estudiante del Atlántico alcance un puntaje global alto en Saber 11 usando únicamente variables familiares. Esta aproximación es útil para la Secretaría de Educación del Atlántico porque permite analizar la relación entre condiciones del hogar y desempeño académico, sin depender de variables institucionales ni de otros puntajes de la prueba.


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)
from sklearn.linear_model import LogisticRegression

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import mlflow
import mlflow.keras

pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)